# AAAC —: Train / Val / Test Split

**Purpose**: Create the canonical `train_aaac.csv`, `val_aaac.csv`, `test_aaac.csv`files that directly load Notebooks 1-5.
**Method**: Stratified 80 / 10 / 10 split on category x language_variant, with fixed seed.
`is_adversarial` is automatically balanced by stratifying on `category`, because every non-`safe` category is adversarial and `safe` is not . Moreover stratifying by `language_variant` guarantees proportional representation for the three scripts (Algerian Derja, Arabizi, Franco-Arabic) in each split, instead of leaving the balance to chance.


## 1. Setup

In [ ]:
!pip install -q scikit-learn pandas

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

# ── Config ──────────────────────────────────────────────────────────────────
DRIVE_BASE = "/content/drive/MyDrive/AAAC"
RAW_PATH   = f"{DRIVE_BASE}/AAAC_dataset.csv"   # upload the raw CSV here first
TRAIN_PATH = f"{DRIVE_BASE}/train_aaac.csv"
VAL_PATH   = f"{DRIVE_BASE}/val_aaac.csv"
TEST_PATH  = f"{DRIVE_BASE}/test_aaac.csv"

TEST_SIZE = 0.10   # 10% of the full dataset
VAL_SIZE  = 0.10   # 10% of the full dataset (so train = 80%)

os.makedirs(DRIVE_BASE, exist_ok=True)
print("Config loaded.")

Config loaded.


## 2. Load Raw Dataset  🔴 T1

In [ ]:
df = pd.read_csv(RAW_PATH)

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print()
print("Missing values per column:")
print(df.isna().sum())
print()
print(f"Duplicate ids:   {df['id'].duplicated().sum()}")
print(f"Duplicate texts: {df['text'].duplicated().sum()}")
print()
print("category counts:")
print(df['category'].value_counts())
print()
print("language_variant counts:")
print(df['language_variant'].value_counts())

Shape: (3090, 6)
Columns: ['id', 'text', 'language_variant', 'category', 'tool', 'is_adversarial']

Missing values per column:
id                  0
text                0
language_variant    0
category            0
tool                0
is_adversarial      0
dtype: int64

Duplicate ids:   0
Duplicate texts: 0

category counts:
category
safe                 1545
prompt_injection      497
hate_speech           372
sensitive_request     368
tool_misuse           308
Name: count, dtype: int64

language_variant counts:
language_variant
arabizi           1260
algerian_derja    1013
franco_arabic      817
Name: count, dtype: int64


## 3. Stratified Train / Val / Test Split

We split the dataset twice using `sklearn.train_test_split`, once to remove the test set and another to split the remaining dataset into train and val. Both splits are stratified on a combined `category__language_variant` key such that each (intent, script) pair is represented in train/val/test in roughly the same proportion as the full dataset. The smallest combination `sensitive_request` x `arabizi` has 92 rows, enough for a clean 80/10/10 split.

In [ ]:
# Sanity check: is_adversarial is fully determined by category (safe → 0, everything else → 1)
check = df.groupby('category')['is_adversarial'].nunique()
assert (check == 1).all(), "category does not cleanly determine is_adversarial — re-examine stratification key"
print("Confirmed: is_adversarial is a deterministic function of category.")
print(df.groupby('category')['is_adversarial'].first())

# Combined stratification key
strata = df['category'].astype(str) + '__' + df['language_variant'].astype(str)

# Stage 1: peel off the test set
train_val_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    stratify=strata,
    random_state=SEED,
)

# Stage 2: split the remainder into train / val
val_fraction_of_remainder = VAL_SIZE / (1 - TEST_SIZE)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=val_fraction_of_remainder,
    stratify=strata.loc[train_val_df.index],
    random_state=SEED,
)

for name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    pct = 100 * len(split_df) / len(df)
    print(f"{name:5s}: {len(split_df):4d} rows  ({pct:.1f}%)")

Confirmed: is_adversarial is a deterministic function of category.
category
hate_speech          1
prompt_injection     1
safe                 0
sensitive_request    1
tool_misuse          1
Name: is_adversarial, dtype: int64
train: 2472 rows  (80.0%)
val  :  309 rows  (10.0%)
test :  309 rows  (10.0%)


## 4. Verification — Leakage & Balance Checks  🔴 T1

In [ ]:
# 4.1 — No id or text leakage across splits
id_sets   = {"train": set(train_df['id']),   "val": set(val_df['id']),   "test": set(test_df['id'])}
text_sets = {"train": set(train_df['text']), "val": set(val_df['text']), "test": set(test_df['text'])}

print("=== id overlap ===")
print(f"train∩val:  {len(id_sets['train']   & id_sets['val'])}")
print(f"train∩test: {len(id_sets['train']   & id_sets['test'])}")
print(f"val∩test:   {len(id_sets['val']     & id_sets['test'])}")

print("\n=== text overlap ===")
print(f"train∩val:  {len(text_sets['train'] & text_sets['val'])}")
print(f"train∩test: {len(text_sets['train'] & text_sets['test'])}")
print(f"val∩test:   {len(text_sets['val']   & text_sets['test'])}")

assert sum(len(id_sets[a] & id_sets[b]) for a, b in [("train","val"),("train","test"),("val","test")]) == 0
print("\n✓ No id or text leakage across splits.")

# 4.2 — Row counts add back up to the original dataset
assert len(train_df) + len(val_df) + len(test_df) == len(df)
print("✓ Split row counts sum to the original dataset size.")

# 4.3 — Per-split distribution, as % of that split (compare against the full-dataset %)
print("\n=== category distribution (% within split) ===")
dist = pd.concat([
    df['category'].value_counts(normalize=True).rename('full'),
    train_df['category'].value_counts(normalize=True).rename('train'),
    val_df['category'].value_counts(normalize=True).rename('val'),
    test_df['category'].value_counts(normalize=True).rename('test'),
], axis=1).round(3)
print(dist)

print("\n=== language_variant distribution (% within split) ===")
dist2 = pd.concat([
    df['language_variant'].value_counts(normalize=True).rename('full'),
    train_df['language_variant'].value_counts(normalize=True).rename('train'),
    val_df['language_variant'].value_counts(normalize=True).rename('val'),
    test_df['language_variant'].value_counts(normalize=True).rename('test'),
], axis=1).round(3)
print(dist2)

=== id overlap ===
train∩val:  0
train∩test: 0
val∩test:   0

=== text overlap ===
train∩val:  0
train∩test: 0
val∩test:   0

✓ No id or text leakage across splits.
✓ Split row counts sum to the original dataset size.

=== category distribution (% within split) ===
                    full  train    val   test
category                                     
safe               0.500  0.500  0.502  0.502
prompt_injection   0.161  0.161  0.159  0.162
hate_speech        0.120  0.121  0.120  0.120
sensitive_request  0.119  0.119  0.120  0.117
tool_misuse        0.100  0.100  0.100  0.100

=== language_variant distribution (% within split) ===
                   full  train    val   test
language_variant                            
arabizi           0.408  0.408  0.408  0.408
algerian_derja    0.328  0.327  0.330  0.330
franco_arabic     0.264  0.265  0.262  0.262


## 5. Save Splits

Writes `train_aaac.csv`, `val_aaac.csv`, `test_aaac.csv` to `DRIVE_BASE`, exactly where
Notebooks 1–5 expect to find them.

In [ ]:
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

train_df.to_csv(TRAIN_PATH, index=False)
val_df.to_csv(VAL_PATH, index=False)
test_df.to_csv(TEST_PATH, index=False)

print("Saved:")
print(f"  {TRAIN_PATH}  ({len(train_df)} rows)")
print(f"  {VAL_PATH}  ({len(val_df)} rows)")
print(f"  {TEST_PATH}  ({len(test_df)} rows)")
print("\nNotebooks 1–5 can now load these files directly.")

Saved:
  /content/drive/MyDrive/AAAC/train_aaac.csv  (2472 rows)
  /content/drive/MyDrive/AAAC/val_aaac.csv  (309 rows)
  /content/drive/MyDrive/AAAC/test_aaac.csv  (309 rows)

Notebooks 1–5 can now load these files directly.
